# SENTINEL Neo4j: Fraud Patterns in a Graph

**Objective.** Fraud often hides in relationships: one phone used by many customers, or a fraudster's device reused by others. We load SENTINEL customers, devices and merchants into Neo4j and use Cypher to find patterns that are hard to see in tables.

| # | Task | Where |
|---|---|---|
| 1 | Run Neo4j (Docker or Aura) and connect from Python with `verify_connectivity()` | Sections 1–3 |
| 2 | Create `Customer`, `Device`, `Merchant` nodes and `USED_DEVICE`, `PAID` relationships (amount, time, fraud_label on `PAID`) from at least 5,000 transactions | Sections 4–8 |
| 3 | Use `MERGE` (not `CREATE`) so the load can run twice, plus uniqueness constraints on the ids | Sections 6–9 |
| 4 | Cypher: devices shared by 3+ customers in 7 days; customers within 2 hops of a fraud payment; merchants most often paid from shared devices | Section 10 |
| 5 | Screenshot one suspicious cluster and explain it | Section 11 |

### Graph model

```
(:Customer)-[:USED_DEVICE {first_seen, last_seen, seen_at, uses}]->(:Device)
(:Customer)-[:PAID {transaction_id, amount, time, fraud_label, device_id}]->(:Merchant)
```

### Setup

**Option A: Neo4j Aura (no install).** Create a free instance at https://console.neo4j.io and save the password it shows.

**Option B: Docker** (needs WSL on Windows). The password must be at least 8 characters:

```
docker run -d --name sentinel-neo4j -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/<your-password> neo4j:5
```

**`.env` file** (never commit it; commit `.env.example` with placeholders):

```
SUPABASE_URL=https://<your-project>.supabase.co
SUPABASE_KEY=<your-key>
NEO4J_URI=neo4j+s://<instance-id>.databases.neo4j.io     # Docker: bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=<your-password>
```

**Packages:** `pip install neo4j supabase python-dotenv pandas`

## 1. Import libraries

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client
from neo4j import GraphDatabase

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

## 2. Load environment variables

In [ ]:
# Load environment variables and retrieve credentials from the .env file
load_dotenv(override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

# Show which values were found (never print the secrets themselves)
for name, value in [("SUPABASE_URL", SUPABASE_URL), ("SUPABASE_KEY", SUPABASE_KEY),
                    ("NEO4J_URI", NEO4J_URI), ("NEO4J_USERNAME", NEO4J_USERNAME), ("NEO4J_PASSWORD", NEO4J_PASSWORD)]:
    print(f"{name:15s} found: {bool(value)}")
print("Neo4j URI:", NEO4J_URI)

## 3. Connect to Supabase and Neo4j

`verify_connectivity()` checks the Neo4j address and password straight away, before any data is loaded. If it fails, the message below lists the usual causes.

In [ ]:
# Connect to Supabase and Neo4j
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Successfully connected to Supabase!")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
try:
    driver.verify_connectivity()
    print("Successfully connected to Neo4j!")
    print("Server:", driver.get_server_info().agent)
except Exception as error:
    print("Could not connect to Neo4j:", type(error).__name__, "-", error)
    print('''
Check:
 1. NEO4J_URI is spelled correctly in .env (NEO4J_URI, not NEO4J_URT) and has no quotes or spaces.
 2. Aura uses neo4j+s://<id>.databases.neo4j.io ; Docker uses bolt://localhost:7687
 3. The Aura instance is running (free instances pause after inactivity: resume it in the console).
 4. NEO4J_USERNAME is "neo4j" and NEO4J_PASSWORD is the one shown when the instance was created.
 5. Restart the kernel after editing .env.''')
    raise

### Helper functions for Cypher

In [ ]:
def run_query(query, **params):
    '''Run a Cypher read query and return the result as a DataFrame.'''
    with driver.session() as session:
        return pd.DataFrame(session.run(query, **params).data())


def run_write(query, **params):
    '''Run a Cypher write query and return Neo4j's counters.'''
    with driver.session() as session:
        return session.run(query, **params).consume().counters


def run_in_batches(query, records, batch_size=1000):
    '''Send records to Neo4j in batches with UNWIND; return totals of what was created.'''
    totals = {"nodes_created": 0, "relationships_created": 0, "properties_set": 0}
    for start in range(0, len(records), batch_size):
        counters = run_write(query, records=records[start:start + batch_size])
        totals["nodes_created"] += counters.nodes_created
        totals["relationships_created"] += counters.relationships_created
        totals["properties_set"] += counters.properties_set
    return totals

## 4. Fetch the data from Supabase

We take a sample of at least **5,000 transactions** in `transaction_id` order. Transaction ids follow time, so the sample is a continuous stretch of activity: sharing patterns such as "3 customers on one device within 7 days" stay intact. Then we fetch only the customers, devices, merchants and labels that these transactions use.

> If Section 10a finds only one or two shared devices, raise `SAMPLE_SIZE` to 10,000–20,000 and run again. A longer stretch of time contains more sharing.

In [ ]:
SAMPLE_SIZE = 5000          # the brief asks for at least 5,000 transactions

# Order each table by its own primary key so pages never overlap or skip rows
ORDER_COLUMNS = {"transactions": "transaction_id", "customers": "customer_id", "devices": "device_id",
                 "merchants": "merchant_id", "labels": "transaction_id"}


# Fetch records from a Supabase table in batches
def fetch_all_rows(table_name, batch_size=1000, limit=None):
    all_rows = []
    start = 0

    while True:
        end = start + batch_size - 1
        if limit is not None:
            end = min(end, limit - 1)

        response = (
            supabase.table(table_name).select("*").order(ORDER_COLUMNS[table_name])
            .range(start, end).execute()
        )

        rows = response.data
        if not rows:
            break

        all_rows.extend(rows)
        start += len(rows)
        if len(rows) < batch_size:
            break

        if limit is not None and len(all_rows) >= limit:
            break
    return all_rows


# Fetch records from a Supabase table that match the given IDs
def fetch_by_ids(table_name, column_name, ids, batch_size=500):
    all_rows = []
    unique_ids = list(set(value for value in ids if value is not None and value == value))
    for i in range(0, len(unique_ids), batch_size):
        batch_ids = unique_ids[i:i + batch_size]
        response = (
            supabase
            .table(table_name)
            .select("*")
            .in_(column_name, batch_ids)
            .execute()
        )
        all_rows.extend(response.data)
    return all_rows

In [ ]:
transactions_raw = fetch_all_rows(table_name="transactions", batch_size=1000, limit=SAMPLE_SIZE)
transactions_df = pd.DataFrame(transactions_raw)
print("Transactions loaded:", len(transactions_df))
transactions_df.head()

In [ ]:
customers_raw = fetch_by_ids("customers", "customer_id", transactions_df["customer_id"])
customers_df = pd.DataFrame(customers_raw)

devices_raw = fetch_by_ids("devices", "device_id", transactions_df["device_id"])
devices_df = pd.DataFrame(devices_raw)

merchants_raw = fetch_by_ids("merchants", "merchant_id", transactions_df["merchant_id"])
merchants_df = pd.DataFrame(merchants_raw)

labels_raw = fetch_by_ids("labels", "transaction_id", transactions_df["transaction_id"])
labels_df = pd.DataFrame(labels_raw)

print("CUSTOMERS:", len(customers_df))
print("DEVICES:", len(devices_df))
print("MERCHANTS:", len(merchants_df))
print("TRANSACTIONS:", len(transactions_df))
print("LABELS:", len(labels_df))
assert len(transactions_df) >= 5000, "The brief asks for at least 5,000 transactions"

## 5. Data preparation

* Attach each payment's `fraud_label` from the labels table.
* Check duplicates, missing values and how the ids match across tables.
* Parse timestamps with `format="ISO8601"`. Supabase returns some timestamps without fractional seconds (`...:44+00:00`) and others with them (`...:44.123+00:00`). Without this option pandas guesses one format and turns the others into invalid dates (`NaT`).
* Build `transactions_for_neo4j`: one plain dictionary per payment, ready to send to Neo4j.

In [ ]:
print("Transaction columns:")
print(transactions_df.columns.tolist())

print("\nLabel columns:")
print(labels_df.columns.tolist())

In [ ]:
transactions_df = transactions_df.merge(labels_df[["transaction_id", "fraud_label"]], on="transaction_id", how="left")
transactions_df["fraud_label"] = transactions_df["fraud_label"].fillna(0).astype(int)
transactions_df[["transaction_id", "fraud_label"]].head()

In [ ]:
print("Duplicate transaction IDs:", transactions_df["transaction_id"].duplicated().sum())

print("\nMissing values:")
print(transactions_df.isnull().sum())

In [ ]:
print("Customers:", customers_df["customer_id"].nunique())
print("Devices:", devices_df["device_id"].nunique())
print("Merchants:", merchants_df["merchant_id"].nunique())

print("\nUnique transaction IDs:", transactions_df["transaction_id"].nunique())
print("Unique customers in transactions:", transactions_df["customer_id"].nunique())
print("Unique devices in transactions:", transactions_df["device_id"].nunique())
print("Unique merchants in transactions:", transactions_df["merchant_id"].nunique())

print("\nFraud labels:")
print(transactions_df["fraud_label"].value_counts(dropna=False))

In [ ]:
# Convert transaction timestamps to UTC datetimes.
# format="ISO8601" accepts timestamps with and without fractional seconds, so none become invalid.
transactions_df["transaction_time_parsed"] = pd.to_datetime(
    transactions_df["transaction_time"], format="ISO8601", errors="coerce", utc=True)
print("Invalid timestamps:", transactions_df["transaction_time_parsed"].isna().sum())

print("\nTransaction time range:")
print("Minimum:", transactions_df["transaction_time_parsed"].min())
print("Maximum:", transactions_df["transaction_time_parsed"].max())

In [ ]:
# Replace missing device IDs with None (Neo4j stores None as "no value")
neo4j_df = transactions_df.copy()
neo4j_df["device_id"] = neo4j_df["device_id"].astype(object).where(neo4j_df["device_id"].notna(), None)

# Prepare transaction records in a dictionary format suitable for inserting into Neo4j
transactions_for_neo4j = []
for row in neo4j_df.to_dict("records"):
    transaction = {
        "transaction_id": row["transaction_id"],
        "customer_id": row["customer_id"],
        "device_id": row["device_id"],
        "merchant_id": row["merchant_id"],
        "amount": float(row["transaction_amount"]),
        "time": (
            row["transaction_time_parsed"].to_pydatetime()
            if pd.notna(row["transaction_time_parsed"])
            else None
        ),
        "fraud_label": int(row["fraud_label"]),
        "channel": row.get("payment_channel"),
    }
    transactions_for_neo4j.append(transaction)

print("Total prepared transactions:", len(transactions_for_neo4j))
print("Transactions with missing device IDs:", sum(t["device_id"] is None for t in transactions_for_neo4j))
print("Transactions with invalid timestamps:", sum(t["time"] is None for t in transactions_for_neo4j))
print("\nSample prepared record:")
print(transactions_for_neo4j[0])

## 6. Create the constraints

* A customer may make hundreds of transactions, but should appear as only **one** `Customer` node.
* A device may be used by several customers, but should appear as only **one** `Device` node.
* A merchant may receive thousands of payments, but should appear as only **one** `Merchant` node.
* Each payment (`PAID.transaction_id`) should exist only once.

A uniqueness constraint makes Neo4j refuse duplicates. It also creates an index, so `MERGE` finds existing nodes quickly.

In [ ]:
# Create uniqueness constraints to prevent duplicate Customer, Device and Merchant nodes
constraints = [
    '''
    CREATE CONSTRAINT customer_id_unique IF NOT EXISTS
    FOR (c:Customer)
    REQUIRE c.customer_id IS UNIQUE
    ''',
    '''
    CREATE CONSTRAINT device_id_unique IF NOT EXISTS
    FOR (d:Device)
    REQUIRE d.device_id IS UNIQUE
    ''',
    '''
    CREATE CONSTRAINT merchant_id_unique IF NOT EXISTS
    FOR (m:Merchant)
    REQUIRE m.merchant_id IS UNIQUE
    ''',
]
for constraint in constraints:
    run_write(constraint)

# One PAID relationship per transaction (relationship constraints need Neo4j 5.7+; else an index)
try:
    run_write('''
    CREATE CONSTRAINT paid_transaction_unique IF NOT EXISTS
    FOR ()-[p:PAID]-()
    REQUIRE p.transaction_id IS UNIQUE
    ''')
except Exception as error:
    print("Relationship constraint not available, creating an index instead:", type(error).__name__)
    run_write("CREATE INDEX paid_transaction_idx IF NOT EXISTS FOR ()-[p:PAID]-() ON (p.transaction_id)")

run_query("SHOW CONSTRAINTS YIELD name, type, labelsOrTypes, properties RETURN name, type, labelsOrTypes, properties")

## 7. Create the nodes with MERGE

`MERGE` finds a node with this id, or creates it if it does not exist. That is why the load can run twice without making duplicates. `SET` then fills in the properties. Records are sent in batches with `UNWIND`, which is much faster than one query per row.

In [ ]:
# Prepare unique records and create Customer nodes in Neo4j using Cypher MERGE
customer_records = (customers_df[["customer_id", "customer_age", "home_city", "region"]]
                    .dropna(subset=["customer_id"]).drop_duplicates("customer_id").to_dict("records"))
CUSTOMER_QUERY = '''
UNWIND $records AS record
MERGE (c:Customer {customer_id: record.customer_id})
SET c.age = record.customer_age, c.home_city = record.home_city, c.region = record.region
'''

# Prepare unique records and create Device nodes
device_records = (devices_df[["device_id", "device_kind"]]
                  .dropna(subset=["device_id"]).drop_duplicates("device_id").to_dict("records"))
DEVICE_QUERY = '''
UNWIND $records AS record
MERGE (d:Device {device_id: record.device_id})
SET d.kind = record.device_kind
'''

# Prepare unique records and create Merchant nodes
merchant_records = (merchants_df[["merchant_id", "synthetic_name", "merchant_category"]]
                    .dropna(subset=["merchant_id"]).drop_duplicates("merchant_id").to_dict("records"))
MERCHANT_QUERY = '''
UNWIND $records AS record
MERGE (m:Merchant {merchant_id: record.merchant_id})
SET m.name = record.synthetic_name, m.category = record.merchant_category
'''


def load_nodes():
    return pd.DataFrame({
        "Customer": run_in_batches(CUSTOMER_QUERY, customer_records),
        "Device": run_in_batches(DEVICE_QUERY, device_records),
        "Merchant": run_in_batches(MERCHANT_QUERY, merchant_records),
    }).T


load_nodes()

## 8. Create the relationships with MERGE

* **`PAID`**: one relationship per payment, merged on `transaction_id`, carrying `amount`, `time`, `fraud_label`, `device_id` and `channel`.
* **`USED_DEVICE`**: one relationship per customer–device pair. It keeps every time the device was used (`seen_at`), adding a time only if it is not already in the list. The 7-day query in Section 10 needs these times, and the check keeps the load idempotent.

In [ ]:
PAID_QUERY = '''
UNWIND $records AS record
MATCH (c:Customer {customer_id: record.customer_id})
MATCH (m:Merchant {merchant_id: record.merchant_id})
MERGE (c)-[p:PAID {transaction_id: record.transaction_id}]->(m)
SET p.amount = record.amount,
    p.time = record.time,
    p.fraud_label = record.fraud_label,
    p.device_id = record.device_id,
    p.channel = record.channel
'''

USED_DEVICE_QUERY = '''
UNWIND $records AS record
WITH record WHERE record.device_id IS NOT NULL AND record.time IS NOT NULL
MATCH (c:Customer {customer_id: record.customer_id})
MATCH (d:Device {device_id: record.device_id})
WITH c, d, record.time AS t
MERGE (c)-[u:USED_DEVICE]->(d)
ON CREATE SET u.seen_at = [t], u.first_seen = t, u.last_seen = t
ON MATCH SET u.seen_at = CASE WHEN t IN u.seen_at THEN u.seen_at ELSE u.seen_at + t END,
             u.first_seen = CASE WHEN t < u.first_seen THEN t ELSE u.first_seen END,
             u.last_seen = CASE WHEN t > u.last_seen THEN t ELSE u.last_seen END
SET u.uses = size(u.seen_at)
'''


def load_relationships():
    return pd.DataFrame({
        "PAID": run_in_batches(PAID_QUERY, transactions_for_neo4j),
        "USED_DEVICE": run_in_batches(USED_DEVICE_QUERY, transactions_for_neo4j),
    }).T


load_relationships()

## 9. Verify the graph, and prove the load can run twice

We count everything, run **exactly the same load again**, and count again. Because every write uses `MERGE`, the second run creates 0 nodes and 0 relationships, and the counts do not change.

In [ ]:
GRAPH_COUNTS = '''
RETURN COUNT { (:Customer) } AS customers,
       COUNT { (:Device) } AS devices,
       COUNT { (:Merchant) } AS merchants,
       COUNT { ()-[:PAID]->() } AS paid,
       COUNT { ()-[:PAID {fraud_label: 1}]->() } AS fraud_paid,
       COUNT { ()-[:USED_DEVICE]->() } AS used_device
'''
counts_first = run_query(GRAPH_COUNTS)
print("Graph after the first load:")
display(counts_first)

# Second run of the same load
second_nodes = load_nodes()
second_relationships = load_relationships()
display(pd.concat([second_nodes, second_relationships]))

counts_second = run_query(GRAPH_COUNTS)
print("Graph after the second load:")
display(counts_second)

created_again = second_nodes["nodes_created"].sum() + second_relationships["relationships_created"].sum()
print("Nodes/relationships created by the second load:", created_again)
print("Counts unchanged:", counts_first.equals(counts_second))

## 10. Fraud patterns in Cypher

### 10a. Devices shared by 3 or more customers within 7 days

Each use of a device starts a 7-day window, and we count the **distinct** customers inside it. A device qualifies when any window holds 3 or more customers. Genuine people rarely share a phone with strangers, so this points to fraud rings, mule handlers, or takeovers of several accounts.

In [ ]:
SHARED_DEVICES_QUERY = '''
MATCH (d:Device)
WHERE COUNT { (d)<-[:USED_DEVICE]-(:Customer) } >= $min_customers
MATCH (c:Customer)-[u:USED_DEVICE]->(d)
UNWIND u.seen_at AS t
WITH d, c.customer_id AS customer, t
ORDER BY t
WITH d, collect({customer: customer, t: t}) AS uses
UNWIND uses AS anchor
WITH d, anchor.t AS window_start,
     [x IN uses WHERE x.t >= anchor.t AND x.t < anchor.t + duration({days: $days}) | x.customer] AS in_window
WITH d, window_start, reduce(s = [], x IN in_window | CASE WHEN x IN s THEN s ELSE s + x END) AS customers
WHERE size(customers) >= $min_customers
WITH d, window_start, customers
ORDER BY size(customers) DESC, window_start
WITH d, collect({start: window_start, customers: customers})[0] AS busiest
OPTIONAL MATCH (c2:Customer)-[p:PAID]->(:Merchant)
WHERE c2.customer_id IN busiest.customers AND p.device_id = d.device_id
RETURN d.device_id AS device_id,
       d.kind AS device_kind,
       size(busiest.customers) AS customers_in_7_days,
       toString(busiest.start) AS window_start,
       busiest.customers AS customers,
       count(p) AS payments_on_device,
       sum(p.fraud_label) AS fraud_payments
ORDER BY customers_in_7_days DESC, fraud_payments DESC, device_id
'''

shared_devices_df = run_query(SHARED_DEVICES_QUERY, min_customers=3, days=7)
print("Devices shared by 3+ customers within 7 days:", len(shared_devices_df))
shared_devices_df.head(20)

**Cross-check with pandas.** The same rule, computed directly from `transactions_df`, must find the same devices.

In [ ]:
def shared_devices_pandas(df, min_customers=3, days=7):
    found = set()
    uses = df.dropna(subset=["device_id", "transaction_time_parsed"]).sort_values("transaction_time_parsed")
    many = uses.groupby("device_id")["customer_id"].transform("nunique") >= min_customers
    for device, group in uses[many].groupby("device_id"):
        times = group["transaction_time_parsed"].to_numpy()
        customers = group["customer_id"].to_numpy()
        for i in range(len(times)):
            inside = (times >= times[i]) & (times < times[i] + pd.Timedelta(days=days))
            if len(set(customers[inside])) >= min_customers:
                found.add(device)
                break
    return found


pandas_devices = shared_devices_pandas(transactions_df)
cypher_devices = set(shared_devices_df["device_id"]) if len(shared_devices_df) else set()
print("pandas:", len(pandas_devices), "| Cypher:", len(cypher_devices), "| identical:", pandas_devices == cypher_devices)

### 10b. Customers within 2 hops of a fraud-labelled payment

We start from every customer who made a payment with `fraud_label = 1`. Two hops **through a device**, *customer → device → customer*, reach everyone who used the same device as a fraudster. These customers are not labelled fraud themselves, but they share hardware with someone who is: a natural watch-list.

Merchants are not used as the middle hop. A merchant is paid by hundreds of customers, so "paid the same shop" would flag almost everyone. The second cell shows how big that naive set is.

In [ ]:
NEAR_FRAUD_QUERY = '''
MATCH (f:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
WITH DISTINCT f
MATCH (f)-[:USED_DEVICE]->(d:Device)<-[:USED_DEVICE]-(other:Customer)
WHERE other <> f
WITH other, collect(DISTINCT f.customer_id) AS linked_fraud_customers, collect(DISTINCT d.device_id) AS via_devices
RETURN other.customer_id AS customer_id,
       size(linked_fraud_customers) AS fraud_customers_linked,
       linked_fraud_customers,
       via_devices,
       COUNT { (other)-[:PAID {fraud_label: 1}]->() } AS own_fraud_payments,
       COUNT { (other)-[:PAID]->() } AS own_payments
ORDER BY fraud_customers_linked DESC, own_fraud_payments DESC, customer_id
'''

near_fraud_df = run_query(NEAR_FRAUD_QUERY)
fraud_customers = run_query("MATCH (f:Customer)-[:PAID {fraud_label: 1}]->() RETURN count(DISTINCT f) AS n")["n"][0]
print("Customers with a fraud payment:", fraud_customers)
print("Other customers within 2 hops (sharing a device with them):", len(near_fraud_df))
near_fraud_df.head(20)

In [ ]:
# For comparison: 2 hops through ANY node, including merchants
NAIVE_QUERY = '''
MATCH (f:Customer)-[:PAID {fraud_label: 1}]->(:Merchant)
WITH DISTINCT f
MATCH (f)--()--(other:Customer)
WHERE other <> f
RETURN count(DISTINCT other) AS customers
'''
total_customers = run_query("MATCH (c:Customer) RETURN count(c) AS n")["n"][0]
naive = run_query(NAIVE_QUERY)["customers"][0]
print(f"Through shared devices only: {len(near_fraud_df)} customers")
print(f"Through devices OR merchants: {naive} of {total_customers} customers ({naive / total_customers:.0%}): too broad to be useful")

### 10c. Merchants most often paid from shared devices

Here a device counts as **shared** when 2 or more customers used it. We follow each shared device to its customers, keep only the payments actually made **from that device** (`p.device_id = d.device_id`), and rank merchants by how many such payments they received.

In [ ]:
MERCHANTS_QUERY = '''
MATCH (d:Device)
WHERE COUNT { (d)<-[:USED_DEVICE]-(:Customer) } >= $min_customers
MATCH (d)<-[:USED_DEVICE]-(c:Customer)-[p:PAID]->(m:Merchant)
WHERE p.device_id = d.device_id
RETURN m.merchant_id AS merchant_id,
       m.name AS merchant_name,
       m.category AS category,
       count(p) AS payments_from_shared_devices,
       count(DISTINCT d) AS shared_devices,
       count(DISTINCT c) AS customers,
       sum(p.fraud_label) AS fraud_payments,
       round(sum(p.amount), 2) AS total_amount
ORDER BY payments_from_shared_devices DESC, fraud_payments DESC
LIMIT 15
'''

top_merchants_df = run_query(MERCHANTS_QUERY, min_customers=2)
top_merchants_df

## 11. One suspicious cluster

We take the most suspicious shared device: the most customers within 7 days, with ties broken by fraud payments. The next cell prints a Cypher query.

1. Paste it into **Neo4j Browser**: http://localhost:7474 for Docker, or the **Query** tab of your Aura instance.
2. Run it and take a screenshot of the graph view.
3. Save it as `screenshots/suspicious_cluster.png` for the README.

In [ ]:
if shared_devices_df.empty:
    print("No device has 3+ customers within 7 days in this sample: falling back to the device with the most customers.")
    cluster_device = run_query('''
    MATCH (d:Device)<-[:USED_DEVICE]-(c:Customer)
    RETURN d.device_id AS device_id, count(c) AS customers ORDER BY customers DESC LIMIT 1
    ''')["device_id"][0]
else:
    cluster_device = shared_devices_df.iloc[0]["device_id"]

print("Paste this into Neo4j Browser:\n")
print(f'''MATCH (d:Device {{device_id: '{cluster_device}'}})<-[u:USED_DEVICE]-(c:Customer)
OPTIONAL MATCH (c)-[p:PAID]->(m:Merchant)
WHERE p.device_id = d.device_id
RETURN d, u, c, p, m''')
print("\nTip: in the result panel, click the PAID label and set its caption to 'fraud_label' or 'amount'.")

In [ ]:
cluster_df = run_query('''
MATCH (d:Device {device_id: $device_id})<-[u:USED_DEVICE]-(c:Customer)
OPTIONAL MATCH (c)-[p:PAID]->(m:Merchant) WHERE p.device_id = d.device_id
RETURN c.customer_id AS customer_id, c.age AS age, c.home_city AS home_city,
       u.uses AS uses_of_device,
       p.transaction_id AS transaction_id, toString(p.time) AS time, p.amount AS amount,
       p.fraud_label AS fraud_label, m.merchant_id AS merchant_id, m.category AS category
ORDER BY time
''', device_id=cluster_device)

cluster_df["time"] = pd.to_datetime(cluster_df["time"], format="ISO8601", utc=True, errors="coerce").dt.tz_convert("Asia/Kolkata")
display(cluster_df)

first, last = cluster_df["time"].min(), cluster_df["time"].max()
fraud_rows = cluster_df[cluster_df["fraud_label"] == 1]
print(f'''
Cluster summary
---------------
Device:            {cluster_device}
Customers:         {cluster_df['customer_id'].nunique()} ({', '.join(sorted(cluster_df['home_city'].dropna().unique()))})
Payments:          {cluster_df['transaction_id'].nunique()} worth ₹{cluster_df['amount'].sum():,.2f}
Time span:         {first:%d %b %Y %H:%M} to {last:%d %b %Y %H:%M} IST ({(last - first).total_seconds() / 86400:.1f} days)
Merchants:         {cluster_df['merchant_id'].nunique()} (categories: {', '.join(cluster_df['category'].dropna().value_counts().index[:4])})
Fraud payments:    {len(fraud_rows)} (customers: {', '.join(fraud_rows['customer_id'].unique()) or 'none'})
Night (00-05 IST): {(cluster_df['time'].dt.hour < 5).sum()} payment(s)
''')

### Explanation of the cluster

*Complete this with the numbers from the cluster summary above.*

**What the graph shows.** One **Device** node sits in the centre, linked by `USED_DEVICE` to several **Customer** nodes. Each customer has `PAID` relationships to **Merchant** nodes, and these are the payments made from this device.

**Why it is suspicious.**
1. **Several customers, one device, a few days.** `[N]` separate customer accounts, from `[cities]`, used the same device within `[X]` days. Genuine customers pay from their own phones. One device behind many accounts is the classic sign of a fraud ring, a mule handler running accounts, or one person operating stolen accounts.
2. **Confirmed fraud on the device.** `[K]` of the payments from this device are labelled fraud. The other customers on the device were not flagged by row-by-row rules, but the graph ties them to the fraud.
3. **Where the money went.** The payments went to `[categories]` merchants. Gift-card, gaming and electronics merchants are where stolen funds are easiest to turn into goods. Night-time payments (midnight–5 AM IST) add to the risk.

**Why tables miss it.** Each transaction row looks ordinary on its own. The pattern only appears when the rows are connected **through the device**, which is exactly what a graph shows at a glance.

**Action.** Send the cluster's customers to manual review, add the device to the blocklist (SENTINEL's `hard_rules` support device blocklists), and add the customers from 10b to a watch-list.

## 12. Close the connection

In [ ]:
driver.close()
print("Neo4j driver closed.")